In [2]:
!pip install -q onnx onnxruntime matplotlib

In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

import onnx
import onnxruntime as ort

from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch Version:", torch.__version__)
print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU")

PyTorch Version: 2.14.1+cpu
Device: cpu
Running on CPU


In [4]:
class CNNModel(nn.Module):

    def __init__(self):
        super(CNNModel, self).__init__()

        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)

        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):

        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))

        x = x.view(x.size(0), -1)

        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x


model = CNNModel().to(device)

print(model)

CNNModel(
  (conv1): Conv2d(1, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (relu): ReLU()
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=1568, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
)


In [5]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = torchvision.datasets.MNIST(
    root="./data",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)

print("Training Images:", len(train_dataset))
print("Testing Images:", len(test_dataset))

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.85MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 155kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.26MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.3MB/s]

Training Images: 60000
Testing Images: 10000


In [6]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 3

for epoch in range(epochs):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{epochs}] "
        f"Loss: {total_loss/len(train_loader):.4f}"
    )

print("Training Completed!")

Epoch [1/3] Loss: 0.1671
Epoch [2/3] Loss: 0.0490
Epoch [3/3] Loss: 0.0349
Training Completed!


In [7]:
model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total

print(f"Model Accuracy: {accuracy:.2f}%")

Model Accuracy: 98.65%


In [8]:
os.makedirs("model_artifacts", exist_ok=True)

checkpoint = {
    "model_name": "CNNModel",
    "architecture": "CNN",
    "input_shape": [1, 1, 28, 28],
    "num_classes": 10,
    "epochs": epochs,
    "accuracy": accuracy,
    "model_state_dict": model.state_dict()
}

torch.save(
    checkpoint,
    "model_artifacts/model_checkpoint.pth"
)

print("Trained checkpoint saved successfully!")

Trained checkpoint saved successfully!


In [9]:
loaded_checkpoint = torch.load(
    "model_artifacts/model_checkpoint.pth",
    map_location=device,
    weights_only=True
)

loaded_model = CNNModel().to(device)

loaded_model.load_state_dict(
    loaded_checkpoint["model_state_dict"]
)

loaded_model.eval()

print("Checkpoint loaded successfully!")
print("Architecture:", loaded_checkpoint["architecture"])
print("Accuracy:", loaded_checkpoint["accuracy"])

Checkpoint loaded successfully!
Architecture: CNN
Accuracy: 98.65


In [10]:
dummy_input = torch.randn(
    1, 1, 28, 28
).to(device)

torch.onnx.export(
    loaded_model,
    dummy_input,
    "model_artifacts/model.onnx",
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    },
    dynamo=False
)

print("ONNX model exported successfully!")

ONNX model exported successfully!


/tmp/ipykernel_1782/2745677870.py:5: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


In [11]:
onnx_model = onnx.load(
    "model_artifacts/model.onnx"
)

onnx.checker.check_model(onnx_model)

print("ONNX Graph Validation: PASSED")

print("\nModel Graph Operations:\n")

for node in onnx_model.graph.node:

    print(
        "Operation:",
        node.op_type,
        "| Name:",
        node.name
    )

print("\nInput:", onnx_model.graph.input[0].name)
print("Output:", onnx_model.graph.output[0].name)

ONNX Graph Validation: PASSED

Model Graph Operations:

Operation: Conv | Name: /conv1/Conv
Operation: Relu | Name: /relu/Relu
Operation: MaxPool | Name: /pool/MaxPool
Operation: Conv | Name: /conv2/Conv
Operation: Relu | Name: /relu_1/Relu
Operation: MaxPool | Name: /pool_1/MaxPool
Operation: Shape | Name: /Shape
Operation: Constant | Name: /Constant
Operation: Gather | Name: /Gather
Operation: Constant | Name: Constant_12
Operation: Unsqueeze | Name: /Unsqueeze
Operation: Constant | Name: /Constant_1
Operation: Concat | Name: /Concat
Operation: Reshape | Name: /Reshape
Operation: Gemm | Name: /fc1/Gemm
Operation: Relu | Name: /relu_2/Relu
Operation: Gemm | Name: /fc2/Gemm

Input: input
Output: output


In [12]:
onnx_session = ort.InferenceSession(
    "model_artifacts/model.onnx",
    providers=["CPUExecutionProvider"]
)

image, label = test_dataset[0]

input_data = image.unsqueeze(0).numpy()

result = onnx_session.run(
    None,
    {"input": input_data}
)

prediction = np.argmax(result[0])

print("Actual Label:", label)
print("ONNX Prediction:", prediction)

if prediction == label:
    print("ONNX Inference: PASSED")
else:
    print("ONNX Inference: COMPLETED (prediction differs)")

Actual Label: 7
ONNX Prediction: 7
ONNX Inference: PASSED


In [13]:
!pip install -q tensorrt-cu12

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [15]:

import tensorrt as trt
import os

onnx_path = "model_artifacts/model.onnx"
engine_path = "model_artifacts/model.engine"

print("TensorRT Version:", trt.__version__)

# Initialize TensorRT
logger = trt.Logger(trt.Logger.WARNING)
builder = trt.Builder(logger)

# Compatibility with different TensorRT versions
if hasattr(trt.NetworkDefinitionCreationFlag, "EXPLICIT_BATCH"):
    network_flags = 1 << int(
        trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH
    )
else:
    network_flags = 0

network = builder.create_network(network_flags)

# Create ONNX parser
parser = trt.OnnxParser(network, logger)

# Read ONNX model
with open(onnx_path, "rb") as model_file:
    success = parser.parse(model_file.read())

if not success:

    print("ONNX Parsing Failed!")

    for i in range(parser.num_errors):
        print(parser.get_error(i))

else:

    print("ONNX Parsing Successful!")

    # Create builder configuration
    config = builder.create_builder_config()

    # Allocate workspace memory
    config.set_memory_pool_limit(
        trt.MemoryPoolType.WORKSPACE,
        1 << 28
    )

    # Handle dynamic batch size
    profile = builder.create_optimization_profile()

    input_name = network.get_input(0).name

    profile.set_shape(
        input_name,
        (1, 1, 28, 28),    # Minimum batch
        (8, 1, 28, 28),    # Optimal batch
        (32, 1, 28, 28)    # Maximum batch
    )

    config.add_optimization_profile(profile)

    print("Building TensorRT Engine...")

    # Build serialized engine
    serialized_engine = builder.build_serialized_network(
        network,
        config
    )

    if serialized_engine is None:

        print("TensorRT Engine Building Failed!")

    else:

        with open(engine_path, "wb") as f:
            f.write(serialized_engine)

        print("TensorRT Engine Created Successfully!")

        print("Engine Location:", engine_path)

        print(
            "Engine Size:",
            round(os.path.getsize(engine_path) / 1024, 2),
            "KB"
        )


TensorRT Version: 11.5.0.63
ONNX Parsing Successful!
Building TensorRT Engine...
TensorRT Engine Building Failed!


In [16]:
import shutil
from google.colab import files

shutil.make_archive(
    "model_artifacts",
    "zip",
    "model_artifacts"
)

files.download("model_artifacts.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>